## ML модели для классификатора пресс релизов ##

In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC 
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_percentage_error as MAPE

df = pd.read_csv('../ML/data_ml.csv')

In [ ]:
X = df[['decision_text', 'key_rate', 'key_rate_prev', 'inflation','inflation_lag_2', 
             'inflation_gap', 'usd_rate', 'text_len_tokens', 'hike_words_ratio', 'cut_words_ratio',
             'hold_words_ratio', 'hike_minus_cut_ratio']]
y = df['decision_future']

In [69]:
print(f'Размер матрицы: {X.shape}')

Размер матрицы: (54, 12)


**Описание датасета**
**decision_text** - категориальный признак, решение по ставке исходя из текущего пресс-релиза: -1 (снижение)/0 (неизменна)/1 (повышение) <br>
**key_rate** - ключевая ставка на данный момент <br>
**key_rate_prev** - ключевая ставка предыдущего пресс-релиза <br>
**inflation** - инфляция на текущий месяц <br>
**inflation_lag_2** - инфляция с лагом два (взято, так как большая корреляция) <br>
**inflation_gap** - разница между инфляций текущей и целевой <br>
**usd_rate** - курс доллара на текущую дату пресс-релиза <br>
**text_len_tokens** - количество токенов в тексте <br>
**hike_words_ratio** - доля слов, которые чаще встречаются перед повышением ставки <br>
**cut_words_ratio** - доля слов, которые чаще встречаются перед понижением ставки <br>
**hold_words_ratio** - доля слов, которые чаще встречаются при сохраненнии ставки<br>
**hike_minus_cut_ratio** - контраст между повышением и понижением <br>
**decision_future** - решение по ставке в будущем -1/0/1 <br>

# Модели SVC и KNN

In [ ]:
# разделяем датасет на трейн и тест
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
# выделяем колонки для стандартизации
stand_col = ['key_rate', 'key_rate_prev', 'inflation', 'inflation_lag_2', 
             'inflation_gap', 'usd_rate', 'text_len_tokens']

# делаем объект масштабирования
scaler = StandardScaler()

# делаем копии, чтобы не трогать текущие данные
Xtrain_scaled = Xtrain.copy()
Xtest_scaled = Xtest.copy()
# считаем параметры стандартизации на трейне и стандартизируем признаки
Xtrain_scaled[stand_col] = scaler.fit_transform(Xtrain[stand_col])
# стандартизируем признаки для тест
Xtest_scaled[stand_col] = scaler.transform(Xtest[stand_col])

In [ ]:
# задаем линейную модель
model = SVC(kernel = 'linear')

# обучаем на трейне
model.fit(Xtrain_scaled, ytrain)

# делаем предсказание на тесте
pred = model.predict(Xtest_scaled)

# считаем метрики
acc_svc = accuracy_score(ytest, pred)
f1_svc = f1_score(ytest, pred, average='weighted')

print("Accuracy:", acc_svc)
print("F1 weighted:", f1_svc)

Accuracy: 0.5
F1 weighted: 0.5111111111111111


Модель дала средние результаты, поэтому следует дальше попробовать изучить другие модели.

In [ ]:
# модель KNN
knn = KNeighborsClassifier()

# параметры для подбора
param_grid = {
    'n_neighbors': range(1, 21),  # количество соседей
    'weights': ['uniform', 'distance'],  # равные веса или взвешенные по расстоянию
    'metric': ['euclidean', 'manhattan']  # тип расстояния
}

# GridSearch с 5-кратной кросс-валидацией и F1 (weighted) как метрика
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1_weighted')

# обучаем модель
grid_search.fit(Xtrain_scaled, ytrain)

# выводим лучшие параметры
print("Лучшие параметры:", grid_search.best_params_)

# выбираем лучшую модель
best_knn = grid_search.best_estimator_

# делаем предсказание на тесте
y_pred = best_knn.predict(Xtest_scaled)

acc_knn = accuracy_score(ytest, y_pred)
f1_knn = f1_score(ytest, y_pred, average='weighted')

# выводим метрики
print("Accuracy:", acc_knn)
print("F1 weighted:", f1_knn)


Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 4, 'weights': 'distance'}
Accuracy: 0.7142857142857143
F1 weighted: 0.7085137085137084


# Модели LogisticRegression и LinearRegression

In [ ]:
# разделяем датасет на трейн и тест
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [54]:
# выделяем колонки для стандартизации
stand_col = ['key_rate', 'key_rate_prev', 'inflation', 'inflation_lag_2', 
             'inflation_gap', 'usd_rate', 'text_len_tokens']

# делаем объект масштабирования
scaler = StandardScaler()

# делаем копии, чтобы не трогать текущие данные
Xtrain_scaled = Xtrain.copy()
Xtest_scaled = Xtest.copy()
# считаем параметры стандартизации на трейне и стандартизируем признаки
Xtrain_scaled[stand_col] = scaler.fit_transform(Xtrain[stand_col])
# стандартизируем признаки для тест
Xtest_scaled[stand_col] = scaler.transform(Xtest[stand_col])

In [55]:
# Создание модели с фиксированными параметрами
model = LogisticRegression()

# Обучение модели
model.fit(Xtrain_scaled, ytrain)

# Оценка модели на тесте
y_pred = model.predict(Xtest_scaled)

acc_lr = accuracy_score(ytest, y_pred)
f1_lr = f1_score(ytest, y_pred, average='micro')

print("Accuracy:", acc_lr)
print("f1:", f1_lr)

Accuracy: 0.6428571428571429
f1: 0.6428571428571429


Общая точность Accuracy = 0.7 показывает, что 70% всех предсказаний модель делает правильно.

F1 метрика = 0.7 — среднее между точностью положительных предсказаний и долей найденных истинных положительных.

То, что Accuracy и F1 примерно совпадают, говорить о том, что модель различает классы при предсказании.

In [56]:
print(y_pred)
print(ytest)

[-1.  0. -1.  1. -1.  0.  0.  1.  0.  0.  1.  1.  1.  1.]
27   -1.0
48    0.0
5    -1.0
1     0.0
8    -1.0
31    0.0
47    0.0
12    0.0
45    1.0
25   -1.0
2     0.0
18    1.0
20    1.0
36    1.0
Name: decision_future, dtype: float64


In [57]:
X2 = df[['decision_text', 'key_rate_prev', 'inflation','inflation_lag_2', 
             'inflation_gap', 'usd_rate', 'text_len_tokens', 'hike_words_ratio', 'cut_words_ratio',
             'hold_words_ratio', 'hike_minus_cut_ratio']]
y2 = df['key_rate']

In [58]:
# Разделение данных
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42)

    # Масштабирование
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

    # Создание модели с фиксированными параметрами
model = LinearRegression()

                # Обучение модели
model.fit(X2_train_scaled, y2_train)

                # Оценка модели на тесте

y2_pred = model.predict(X2_test_scaled)

In [59]:
mape = MAPE(y2_test, y2_pred)*100
# Вывод результатов
print(f"MAPE: {mape:.1f}%")

MAPE: 1.1%


в среднем предсказанные значения отклоняются от истинных на ~1.1%. То есть,ошибка около 15–16% от реального.

In [60]:
print(y2_pred)
print(y2_test)

[ 7.39496725 20.74285219 20.60111146  4.09482935 18.20128583  7.27988804
  6.47271387 17.95586187  7.80400649  7.48966202  4.21706788]
19     7.50
49    21.00
48    21.00
12     4.25
44    18.00
5      7.25
17     6.50
52    18.00
3      7.75
32     7.50
13     4.25
Name: key_rate, dtype: float64


## Выводы работы

In [68]:
acc = {'SVC': acc_svc, 'KNN': acc_knn, 'Logistic Regression': acc_lr}
f1 = {'SVC': f1_svc, 'KNN': f1_knn, 'Logistic Regression': f1_lr}

df_metrics = pd.DataFrame({
    'Accuracy': acc,
    'F1': f1
})

print('Метрики качества моделей')
df_metrics

Метрики качества моделей


,Accuracy,F1
SVC,0.500000,0.511111
KNN,0.714286,0.708514
Logistic Regression,0.642857,0.642857


1. **SVC** работает плохо, предсказывает только половину.
Это объясняется высокой чувствительностью SVC к масштабам признаков и параметрам. 
Несмотря на стандартизацию числовых признаков, SVC плохо работает с небольшими объёмами выборки, слабой линейной разделимостью классов, шумными текстовыми фичами.
Скорее всего на 54 строках модель не может построить устойчивую разделяющую поверхность.

2. **KNN** показывает лучшую метрику
KNN хорошо работает на небольших датасетах так как опирается на ближайшие примеры.
Модель также невосприимчив к линейности.
Но KNN склонна к переобучению, поэтому стоит проверить на кросс-валидации

3. **Logistic Regression** показывает средний результат
Логистическая регрессия ищет линейную границу между классами, но решение ЦБ — нелинейная функция от макроэкономики, текстовые признаки тоже не образуют линейно разделяемое пространство.
Используются недостаточно данных о тексте, что упрощает представление текста поэтому линейная модель не получает достаточного количества сигналов.

**Что можно добавить:**
- увеличить датасет за счет пересмотра решений, так как большая часть пресс-релизов отсеилась по данному признаку
- добавить TF-IDF признаки
- применить кросс-валидацию
- настроить гиперпараметры в моделях